# Face Recognition — Base d'embeddings InsightFace (LFW Top-20)
**Auteur** : Ibrahima Gabar Diop — PFE Sonatel Academy


In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'insightface', 'onnxruntime-gpu', '-q'], check=True)


In [ ]:
import os, shutil, random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from insightface.app import FaceAnalysis

OUTPUT_DIR = Path('/kaggle/working')
REFS_DIR   = OUTPUT_DIR / 'face_refs'
REFS_DIR.mkdir(parents=True, exist_ok=True)

N_CLASSES         = 20
MIN_IMAGES        = 30
N_REFS_PER_PERSON = 5
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Trouver le CSV et le dossier d'images peu importe la structure de montage
print('Recherche dans /kaggle/input/ ...')
csv_files = list(Path('/kaggle/input').rglob('*allnames*.csv'))
jpg_dirs  = list(Path('/kaggle/input').rglob('George_W_Bush'))
print(f'  CSV trouves  : {csv_files}')
print(f'  George_W_Bush: {jpg_dirs}')

assert csv_files, 'CSV lfw_allnames.csv introuvable !'
assert jpg_dirs,  'Dossier George_W_Bush introuvable !'

CSV_PATH = csv_files[0]
LFW_DIR  = jpg_dirs[0].parent   # dossier parent = racine des personnes
print(f'CSV_PATH : {CSV_PATH}')
print(f'LFW_DIR  : {LFW_DIR}')
print(f'Exemple  : {next(LFW_DIR.rglob("*.jpg"))}')


## 1. Selection des classes


In [ ]:
df_all = pd.read_csv(CSV_PATH)
top20  = df_all[df_all['images'] >= MIN_IMAGES].sort_values('images', ascending=False).head(N_CLASSES)
print(top20[['name','images']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top20['name'][::-1], top20['images'][::-1], color='steelblue')
ax.set_xlabel("Nombre d'images")
ax.set_title(f'LFW Top-{N_CLASSES}')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_balance.png', dpi=150)
plt.show()


## 2. InsightFace buffalo_l


In [ ]:
face_app = FaceAnalysis(
    name='buffalo_l',
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)
face_app.prepare(ctx_id=0, det_size=(640, 640))
print('InsightFace buffalo_l charge (RetinaFace + ArcFace ResNet50)')


## 3. Generation des embeddings


In [ ]:
embeddings_db = {}
failed = []

for _, row in top20.iterrows():
    person     = row['name']
    person_dir = LFW_DIR / person
    imgs       = sorted(person_dir.glob('*.jpg'))
    if not imgs:
        failed.append(person)
        print(f'  SKIP  {person}')
        continue

    random.shuffle(imgs)
    sample = imgs[:N_REFS_PER_PERSON]
    embs, best_img_path, best_area = [], None, 0

    for img_path in sample:
        img   = cv2.imread(str(img_path))
        faces = face_app.get(img)
        if not faces:
            continue
        face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
        embs.append(face.embedding)
        area = (face.bbox[2]-face.bbox[0])*(face.bbox[3]-face.bbox[1])
        if area > best_area:
            best_area, best_img_path = area, img_path

    if not embs:
        failed.append(person)
        print(f'  FAIL  {person}')
        continue

    mean_emb = np.mean(embs, axis=0)
    mean_emb /= np.linalg.norm(mean_emb)
    embeddings_db[person] = mean_emb
    shutil.copy(best_img_path, REFS_DIR / f'{person}.jpg')
    print(f'  OK    {person:35s} ({len(embs)} embs)')

print(f'\nTotal : {len(embeddings_db)} OK, {len(failed)} echecs : {failed}')


## 4. Export


In [ ]:
npz_path = OUTPUT_DIR / 'face_embeddings.npz'
np.savez(str(npz_path), **embeddings_db)
print(f'face_embeddings.npz : {npz_path.stat().st_size/1024:.0f} KB')
print(f'face_refs/          : {len(list(REFS_DIR.glob("*.jpg")))} images')
loaded = np.load(str(npz_path))
print(f'Verification : dim={loaded[list(loaded.keys())[0]].shape}, noms={list(loaded.keys())[:3]}...')


In [ ]:
ref_imgs = sorted(REFS_DIR.glob('*.jpg'))
cols = 5
rows = max(1, (len(ref_imgs) + cols - 1) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(15, rows*3))
axes = axes.flatten()
for i, img_path in enumerate(ref_imgs):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].set_title(img_path.stem[:18], fontsize=8)
    axes[i].axis('off')
for j in range(i+1, len(axes)):
    axes[j].axis('off')
plt.suptitle('Images de reference', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'reference_faces.png', dpi=150)
plt.show()


In [ ]:
print('=' * 50)
print('  DONE — Telecharger depuis /kaggle/working :')
print('  face_refs/*.jpg     -> data/faces/')
print('  face_embeddings.npz -> data/ (optionnel)')
print('=' * 50)
